# Part 4: Code your own diffusion models

In [ ]:
import functools
import os
import random
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
from tqdm.notebook import trange, tqdm
from torchvision.datasets import FashionMNIST
from torch.optim import Adam
from torch.utils.data import DataLoader

In [ ]:
device = 'cuda'

References:

- [[Song21]](https://openreview.net/pdf?id=PxTIG12RRHS) Score-Based Generative Modeling Through Stochastic Differential Equations
- Yang Song's blog

## Variance Preserving SDE (VP)

Variance Preserving diffusion SDE (Eq 11 in [Song21]):

\begin{align*}
d \mathbf{x} = - \frac{1}{2} \beta(t) \mathbf{x} dt + \sqrt{\beta(t)}d\mathbf{w}
\end{align*}

This follows the general SDE form $d \mathbf{x} = f(\mathbf{x}, t) dt + g(t) d \mathbf{w}$. We call $f(\mathbf{x}, t)$ the **drift coefficient** and $g(t)$ the **diffusion coefficient**.

The corresponding general conditional linear Gaussian distribution is:
\begin{align*}
p(x_0|x_t) = \mathcal{N}(x_t; \alpha(t)x_0, \sigma^2(t)I)
\end{align*}
where $\alpha: [0,1] \rightarrow \mathbb{R}$,  $\sigma: [0,1] \rightarrow \mathbb{R}$

$\mu(t), \sigma(t)$ can be derived analytically from $f(\mathbf{x}, t), g(t)$.

\begin{align*}
\begin{cases}
  \mu(t) = \alpha(t)x_0 = \exp{(c(t))}x_0 \\
  \sigma^2(t) = 1 - \exp(2c(t))
\end{cases}
\end{align*}

### Implementing the VP SDE

Refer to Equations (32) and (33) in [Song21] to identify:
* $\beta(t)$
* $c(t)$
* $\mu(t)$
* $\sigma(t)$

Fill in the TODOs in the VP class accordingly

In [ ]:
class VP():
    def __init__(self, beta_min, beta_max, num_steps):
        self.beta_0 = beta_min
        self.beta_1 = beta_max
        self.num_steps = num_steps
        self.discrete_betas = torch.linspace(beta_min / num_steps, beta_max / num_steps, num_steps)
        self.alphas = 1. - self.discrete_betas

    def _beta_t(self, t):
        # TODO: compute beta(t)
        beta_t = self.beta_0 + (self.beta_1 - self.beta_0) * t
        return beta_t

    def _c_t(self, t):
        # TODO: compute c(t)
        # log of signal scaling: c(t) = -0.5 * integral_0^t beta(s) ds
        c_t = -0.5 * (self.beta_0 * t + 0.5 * (self.beta_1 - self.beta_0) * t ** 2)
        return c_t

    def marginal_proba(self, x, t):
        """ Compute the mean and standard deviation of the marginal prob p(x_0|x_t)
        """
        # TODO: compute mu and std (std is a scalar)
        # ensure t is at least 1-d so [:, None, None, None] indexing works
        if not isinstance(t, torch.Tensor):
            t = torch.tensor([float(t)])
        if t.dim() == 0:
            t = t.unsqueeze(0)
        c = self._c_t(t)
        mu_t = torch.exp(c)[:, None, None, None] * x
        std_t = torch.sqrt(1 - torch.exp(2 * c))
        return mu_t, std_t

    def drift(self, x, t):
        """ Compute the VP drift coefficient f(x, t)
        """
        # TODO: compute drift coefficient -- make sure to give beta_t the appropriate shape
        beta_t = self._beta_t(t)[:, None, None, None]
        drift = -0.5 * beta_t * x
        return drift

    def diffusion(self, t):
        """ Compute the VP diffusion coefficient g(t)
        """
        # TODO: compute diffusion coefficient
        diffusion = torch.sqrt(self._beta_t(t))
        return diffusion

## Sampling with VP sde

As per Appendix E of [Song21], recall that for any SDE of the form
\begin{align*}
d \mathbf{x} = \mathbf{f}(\mathbf{x}, t) dt + g(t) d\mathbf{w},
\end{align*}
the reverse-time SDE is given by
\begin{align*}
d \mathbf{x} = [\mathbf{f}(\mathbf{x}, t) - g(t)^2 \nabla_\mathbf{x} \log p_t(\mathbf{x})] dt + g(t) d \bar{\mathbf{w}}.
\end{align*}

We use the [Euler-Maruyama](https://en.wikipedia.org/wiki/Euler%E2%80%93Maruyama_method) numerical method to solve for the reverse-time SDE. This method relies on discretizing the SDE, replacing $dt$ with $\Delta t$ and $d \mathbf{w}$ with $\mathbf{z} \sim \mathcal{N}(\mathbf{0}, g^2(t) \Delta t \mathbf{I})$.

This lead to the following iteration rule:
\begin{align}
\mathbf{x}_{t-\Delta t} =
\mathbf{x}_t
- \mathbf{f}(\mathbf{x}_t, t)\Delta t
+ g^2(t) s_\theta(\mathbf{x}_t, t)\Delta t
+ g(t)\sqrt{\Delta_t}\mathbf{z}_t.
\end{align}

Note: for the last step (i.e. $t-\Delta_t = 0$), we do not wish to add back noise ($g(t)\sqrt{\Delta_t}\mathbf{z}_t$).

In [ ]:
def Euler_Maruyama_sampler(score_model,
                           sde,
                           batch_size,
                           num_steps=1000,
                           device='cuda',
                           eps=1e-3):
    # sample std at t=1
    t1 = torch.ones(batch_size, device=device)
    _, std = sde.marginal_proba(torch.zeros(batch_size, 1, 28, 28, device=device), t1)

    # sample a batch of x at t=1
    init_x = torch.randn(batch_size, 1, 28, 28, device=device) * std[:, None, None, None]

    # create a sequence of time_steps from 1 to eps
    time_steps = torch.linspace(1., eps, num_steps, device=device)
    step_size = time_steps[0] - time_steps[1]

    x = init_x
    with torch.no_grad():
        for time_step in tqdm(time_steps):
            batch_t = torch.ones(batch_size, device=device) * time_step
            score = score_model(x, batch_t)
            g_t = sde.diffusion(batch_t)[:, None, None, None]
            mean = x - sde.drift(x, batch_t) * step_size + g_t ** 2 * score * step_size
            x_ = mean
            x = mean + g_t * torch.sqrt(step_size) * torch.randn_like(x)
    # Do not include any noise in the last sampling step.
    return x_

In [ ]:
def predictor_corrector_sampler(score_model,
                                sde,
                                batch_size,
                                num_steps=1000,
                                device='cuda',
                                snr=0.16,
                                num_corrector_steps=1,
                                eps=1e-3):
    # sample std at t=1
    t1 = torch.ones(batch_size, device=device)
    _, std = sde.marginal_proba(torch.zeros(batch_size, 1, 28, 28, device=device), t1)

    # sample a batch of x at t=1
    init_x = torch.randn(batch_size, 1, 28, 28, device=device) * std[:, None, None, None]

    # create a sequence of time_steps from 1 to eps
    time_steps = torch.linspace(1., eps, num_steps, device=device)
    step_size = time_steps[0] - time_steps[1]

    x = init_x
    with torch.no_grad():
        for time_step in tqdm(time_steps):
            batch_t = torch.ones(batch_size, device=device) * time_step
            int_time_step = int(time_step * (sde.num_steps - 1))
            alpha_t = sde.alphas[int_time_step]

            # Corrector step (Langevin MCMC - alorithm 5 in [Song21])
            for j in range(num_corrector_steps):
                score = score_model(x, batch_t)
                noise = torch.randn_like(x)
                score_norm = torch.norm(score.reshape(batch_size, -1), dim=-1).mean()
                noise_norm = torch.norm(noise.reshape(batch_size, -1), dim=-1).mean()
                step = 2 * alpha_t * (snr * noise_norm / (score_norm + 1e-8)) ** 2
                x = x + step * score + torch.sqrt(2 * step) * noise

            # Predictor step (Euler-Maruyama)
            score = score_model(x, batch_t)
            g_t = sde.diffusion(batch_t)[:, None, None, None]
            mean = x - sde.drift(x, batch_t) * step_size + g_t ** 2 * score * step_size
            x_ = mean
            x = mean + g_t * torch.sqrt(step_size) * torch.randn_like(x)

    # Do not include any noise in the last sampling step.
    return x_

## Setup -- no TODOs to fill in here :)

### Config

In [ ]:
n_epochs = 50
batch_size = 64
lr = 1e-4
num_steps=1000
checkpoint_dir = './checkpoints/'

### Dataset

In [ ]:
# training dataset
train_transforms = transforms.Compose([transforms.ToTensor()])
train_dataset = FashionMNIST('.', train=True, transform=train_transforms, download=True);
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)

### Score-matching model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class GaussianFourierProjection(nn.Module):
    """Gaussian random features for encoding time steps."""
    def __init__(self, embed_dim, scale=30.):
        super().__init__()
        # Randomly sample weights during initialization. These weights are fixed
        # during optimization and are not trainable.
        self.W = nn.Parameter(torch.randn(embed_dim // 2) * scale, requires_grad=False)
    def forward(self, x):
        x_proj = x[:, None] * self.W[None, :] * 2 * np.pi
        return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)


class Dense(nn.Module):
    """A fully connected layer that reshapes outputs to feature maps."""
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.dense = nn.Linear(input_dim, output_dim)
    def forward(self, x):
        return self.dense(x)[..., None, None]


class ScoreNet(nn.Module):
    """A time-dependent score-based model built upon U-Net architecture."""

    def __init__(self, marginal_proba, channels=[32, 64, 128, 256], embed_dim=256):
        """Initialize a time-dependent score-based network.

        Args:
          marginal_proba: A function that takes time t and gives the standard
            deviation of the perturbation kernel p_{0t}(x(t) | x(0)).
          channels: The number of channels for feature maps of each resolution.
          embed_dim: The dimensionality of Gaussian random feature embeddings.
        """
        super().__init__()
        # Gaussian random feature embedding layer for time
        self.embed = nn.Sequential(GaussianFourierProjection(embed_dim=embed_dim),
             nn.Linear(embed_dim, embed_dim))
        # Encoding layers where the resolution decreases
        self.conv1 = nn.Conv2d(1, channels[0], 3, stride=1, bias=False)
        self.dense1 = Dense(embed_dim, channels[0])
        self.gnorm1 = nn.GroupNorm(4, num_channels=channels[0])
        self.conv2 = nn.Conv2d(channels[0], channels[1], 3, stride=2, bias=False)
        self.dense2 = Dense(embed_dim, channels[1])
        self.gnorm2 = nn.GroupNorm(32, num_channels=channels[1])
        self.conv3 = nn.Conv2d(channels[1], channels[2], 3, stride=2, bias=False)
        self.dense3 = Dense(embed_dim, channels[2])
        self.gnorm3 = nn.GroupNorm(32, num_channels=channels[2])
        self.conv4 = nn.Conv2d(channels[2], channels[3], 3, stride=2, bias=False)
        self.dense4 = Dense(embed_dim, channels[3])
        self.gnorm4 = nn.GroupNorm(32, num_channels=channels[3])

        # Decoding layers where the resolution increases
        self.tconv4 = nn.ConvTranspose2d(channels[3], channels[2], 3, stride=2, bias=False)
        self.dense5 = Dense(embed_dim, channels[2])
        self.tgnorm4 = nn.GroupNorm(32, num_channels=channels[2])
        self.tconv3 = nn.ConvTranspose2d(channels[2] + channels[2], channels[1], 3, stride=2, bias=False, output_padding=1)
        self.dense6 = Dense(embed_dim, channels[1])
        self.tgnorm3 = nn.GroupNorm(32, num_channels=channels[1])
        self.tconv2 = nn.ConvTranspose2d(channels[1] + channels[1], channels[0], 3, stride=2, bias=False, output_padding=1)
        self.dense7 = Dense(embed_dim, channels[0])
        self.tgnorm2 = nn.GroupNorm(32, num_channels=channels[0])
        self.tconv1 = nn.ConvTranspose2d(channels[0] + channels[0], 1, 3, stride=1)

        # The swish activation function
        self.act = lambda x: x * torch.sigmoid(x)
        self.marginal_proba = marginal_proba

    def forward(self, x, t):
        # Obtain the Gaussian random feature embedding for t
        embed = self.act(self.embed(t))
        # Encoding path
        h1 = self.conv1(x)
        ## Incorporate information from t
        h1 += self.dense1(embed)
        ## Group normalization
        h1 = self.gnorm1(h1)
        h1 = self.act(h1)
        h2 = self.conv2(h1)
        h2 += self.dense2(embed)
        h2 = self.gnorm2(h2)
        h2 = self.act(h2)
        h3 = self.conv3(h2)
        h3 += self.dense3(embed)
        h3 = self.gnorm3(h3)
        h3 = self.act(h3)
        h4 = self.conv4(h3)
        h4 += self.dense4(embed)
        h4 = self.gnorm4(h4)
        h4 = self.act(h4)

        # Decoding path
        h = self.tconv4(h4)
        ## Skip connection from the encoding path
        h += self.dense5(embed)
        h = self.tgnorm4(h)
        h = self.act(h)
        h = self.tconv3(torch.cat([h, h3], dim=1))
        h += self.dense6(embed)
        h = self.tgnorm3(h)
        h = self.act(h)
        h = self.tconv2(torch.cat([h, h2], dim=1))
        h += self.dense7(embed)
        h = self.tgnorm2(h)
        h = self.act(h)
        h = self.tconv1(torch.cat([h, h1], dim=1))

        # Normalize output
        _, std = self.marginal_proba(x, t)
        h = h / std[:, None, None, None]
        return h

### Loss function

In [ ]:
def loss_fn(model, x, sde, eps=1e-5):
    """ Inputs:
          model: score model (i.e. diffusion model)
          x: batch of images
          sde: instance of VP class
          eps: parameter for numerical stability (1e-5 for learning, 1e-3 for sampling)
    """
    random_t = torch.rand(x.shape[0], device=x.device) * (1. - eps) + eps
    z = torch.randn_like(x)
    mean, std = sde.marginal_proba(x, random_t)
    perturbed_x = mean + z * std[:, None, None, None]

    # predict the score function for each perturbed x in the batch and its corresponding random t
    score = model(perturbed_x, random_t)

    # compute loss
    losses = score * std[:, None, None, None] + z
    loss = torch.mean(torch.sum(losses**2, dim=(1,2,3)))
    return loss

## Train the score model

In [ ]:
def train(sde_params):
    # VP sde
    beta_min, beta_max = sde_params
    sde = VP(beta_min, beta_max, num_steps)

    score_model = torch.nn.DataParallel(ScoreNet(marginal_proba=sde.marginal_proba))
    score_model = score_model.to(device)
    optimizer = Adam(score_model.parameters(), lr=lr)

    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)
    params_str = '{}_{}'.format(beta_min, beta_max)
    checkpoint_path = checkpoint_dir+'ckpt_{}_{}epochs_{}.pth'.format("fashionmnist", n_epochs, params_str)

    # load checkpoint if existing
    if os.path.exists(checkpoint_path):
        score_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        print("model loaded from checkpoint!")
    # otherwise train from scratch
    else:
        losses = []
        patience = 0
        best_loss = float('inf')
        tqdm_epoch = trange(n_epochs)
        for epoch in tqdm_epoch:
            avg_loss = 0.
            num_items = 0
            for x, y in train_loader:
                x = x.to(device)
                loss = loss_fn(score_model, x, sde)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                avg_loss += loss.item() * x.shape[0]
                num_items += x.shape[0]
            epoch_loss = avg_loss / num_items
            tqdm_epoch.set_description('Average Loss: {:5f}'.format(epoch_loss))
            losses.append(epoch_loss)

            # save model state dictionary when loss improves by at least 0.5
            if len(losses) >= 2 and losses[-2] - losses[-1] >= 0.5:
                torch.save(score_model.state_dict(), checkpoint_path)
                patience = 0
            else:
                patience += 1

            # early stopping: stop if no significant improvement for 3 epochs
            if patience >= 3:
                # save final checkpoint before stopping
                torch.save(score_model.state_dict(), checkpoint_path)
                break

        # Plot losses on log scale
        plt.figure()
        plt.plot(range(len(losses)), losses)
        plt.yscale('log')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Training loss (log scale)')
        plt.show()
    return score_model

In [ ]:
sde_params = [0.1, 20.0]
train(sde_params)

## Generation of new samples

In [ ]:
from torchvision.utils import make_grid
def plot_images(images):
    sample_grid = make_grid(images, nrow=int(np.sqrt(images.shape[0])))
    plt.figure(figsize=(6,6))
    plt.axis('off')
    plt.imshow(sample_grid.cpu().permute(1, 2, 0).squeeze())
    plt.show()

In [ ]:
def setup_for_sampling(sde_params):
    beta_min, beta_max = sde_params
    params_str = '{}_{}'.format(beta_min, beta_max)
    checkpoint_path = checkpoint_dir+'ckpt_{}_{}epochs_{}.pth'.format("fashionmnist", n_epochs, params_str)

    sde = VP(beta_min, beta_max, num_steps)
    score_model = torch.nn.DataParallel(ScoreNet(marginal_proba=sde.marginal_proba))
    score_model = score_model.to(device)
    score_model.eval()
    score_model.load_state_dict(torch.load(checkpoint_path, map_location=device))

    return sde, score_model

### Sample

In [ ]:
sde_params = [0.1, 20.0]
sde, score_model = setup_for_sampling(sde_params)

In [ ]:
samples = Euler_Maruyama_sampler(score_model, sde, batch_size, num_steps=1000)
plot_images(samples)

In [ ]:
samples = predictor_corrector_sampler(score_model, sde, batch_size, num_steps=1000)
plot_images(samples)

# Part 4B: Diffusion models for inverse problems

## EC (10pts) : inpainting -- keep until you're done with the pset
This part will require you to read and understand [[Song22]](https://arxiv.org/pdf/2111.08005.pdf) Solving Inverse Problems In Medical Imaging With Score-Based Generative Models.

In our toy example, our measurement matrix A will be an explicit inpainting matrix that replaces some of the pixels in an image by 0 (blackening them out).

Your job will be to modify the sampling process to include conditioning on the perturbed measurements at various t.

**Important: do not attempt until you are entirely satisfied with your work for all of the rest of the assignment (Parts 1 through 5).**

**Important: Minimum guidance will be provided in OH or on Piazza. 🎶🎵You're on your own, kid. You always have been🎵🎶**

**Expectations:** The expected result is a plot with 3 rows: clean images, inpainted images, reconstructed images. The plot **must** include the PSNR of each reconstructed image. You should use at least 5 clean images. We wrote the plotting function for you. We also expect you to explain and justify your implementation choices. Everything (plot, explanations, justifications) **must** appear in your submission PDF.

*(hopefully this scratches the itch of those asking for "moar coding" :))*

In [ ]:
# helper function to inpaint with 0 pixels and get the subsampling matrix
def inpaint(images, ratio=0.05):
    num_pixels = images.shape[-2] * images.shape[-1]
    num_samples = int(ratio * num_pixels)
    # create subsampling matrix A
    A = torch.eye(num_pixels, device=images.device)
    for pixel in random.sample(range(0, num_pixels), num_samples):
        A[pixel][pixel] = 0
    # black out pixels in images using A (a binary matrix with zeroes where we want to black out pixels)
    inpainted_images = images.clone()
    for i in range(len(images)):
        inpainted_images[i] = torch.reshape(torch.matmul(A, inpainted_images[i].view(num_pixels)), images[0].shape)
    return inpainted_images, A

In [ ]:
# helper function to compute the peak signal-to-noise ratio (PSNR)
def psnr(clean, noisy):
    # our range of values is [0.,1.]
    eps = 1e-8
    mse = torch.mean((clean.float() - noisy.float()) ** 2)
    psnr = 10 * torch.log10(1.0 / (mse + eps))
    return psnr

In [ ]:
# helper function to plot samples
def plot_before_after(clean_images, imgs_before, imgs_after, title=""):
    assert(imgs_before.shape[0] == imgs_after.shape[0])
    fig, axs = plt.subplots(3, imgs_before.shape[0], figsize=(16, 10))
    # plot 3 rows: clean, then subsampled, then denoised
    for i, images in enumerate([clean_images, imgs_before, imgs_after]):
        for j, image in enumerate(images):
            axs[i][j].imshow(image.cpu().permute(1, 2, 0).squeeze())
            axs[i][j].set_xticks([])
            axs[i][j].set_yticks([])
    # compute PSNR
    for j, image in enumerate(imgs_after):
        clean = clean_images[j].cpu().permute(1,2,0).squeeze()
        noisy = image.cpu().permute(1,2,0).squeeze()
        psnr_val = psnr(clean, noisy).item()
        axs[2][j].set_title('PSNR: {:.3f}'.format(psnr_val), y=-0.2)
    fig.suptitle(title, size=20)

In [ ]:
# helper functions to condition the reverse diffusion
def get_y_t(images, t, marginal_proba):
    # vector of t
    ts = t * torch.ones(images.shape[0], device=images.device)
    ts = ts[:, None, None, None]

    # sample some noise
    z = torch.randn_like(images)

    # perturb at level t
    _, std = marginal_proba(x=0, t=t)
    perturbed_images = images + z * std
    return perturbed_images

def lbda_scheduler(t, lbda, param):
    param = torch.tensor(param)
    f_t = param*t
    lbda = lbda * f_t
    return lbda

def condition_on_inpainted_y(raw_images, x_t, t, marginal_prob_std, A, lbda=.01, lbda_param=10):
    y_t = get_y_t(raw_images, t, marginal_prob_std)
    lbda = lbda_scheduler(t, lbda, param=lbda_param)
    P, T = [torch.eye(raw_images.shape[-2] * raw_images.shape[-1],device=x_t.device)] * 2
    A = A
    # turn images into column vectors
    flat_y_t = torch.unsqueeze(torch.flatten(y_t, start_dim=1), dim=2)
    flat_x_t = torch.unsqueeze(torch.flatten(x_t, start_dim=1), dim=2)
    # x_prime is a weighted function of x and y
    y_influence = lbda * torch.matmul(A, torch.matmul(torch.inverse(P), flat_y_t))
    x_influence = (1 - lbda) * torch.matmul(A, torch.matmul(T, flat_x_t)) + \
                  torch.matmul(torch.eye(A.shape[0], device=A.device) - A,
                               torch.matmul(T, flat_x_t))
    x_t_prime = torch.reshape(y_influence + x_influence, x_t.shape)
    return x_t_prime

In [ ]:
# Inpainted images
num_images = 8
data, _ = next(iter(train_loader))
clean_images = data[:num_images].to(device)
corrupted_images, A = inpaint(clean_images, ratio=0.75)

# Run EM sampler conditioned on inpainted measurements at each step
sde_inp, score_model_inp = setup_for_sampling(sde_params)
t1 = torch.ones(num_images, device=device)
_, std_1 = sde_inp.marginal_proba(torch.zeros(num_images, 1, 28, 28, device=device), t1)
x = torch.randn(num_images, 1, 28, 28, device=device) * std_1[:, None, None, None]
inp_steps = torch.linspace(1., 1e-3, 500, device=device)
inp_step_size = inp_steps[0] - inp_steps[1]

with torch.no_grad():
    for time_step in tqdm(inp_steps):
        batch_t = torch.ones(num_images, device=device) * time_step
        score = score_model_inp(x, batch_t)
        g_t = sde_inp.diffusion(batch_t)[:, None, None, None]
        mean = x - sde_inp.drift(x, batch_t) * inp_step_size + g_t ** 2 * score * inp_step_size
        x_clean = mean
        x = mean + g_t * torch.sqrt(inp_step_size) * torch.randn_like(x) if time_step > 1e-3 else mean
        x = condition_on_inpainted_y(clean_images, x, time_step, sde_inp.marginal_proba, A)

recovered_images = x_clean

# Expected plot
plot_before_after(clean_images, corrupted_images, recovered_images, title="Inverting 75% inpainting on FashionMNIST")

# Part 5: Diffusion models on a larger scale

As mentioned in the write-up, you only need to include your code for plotting in this notebook for part 5. Here we provide some helper code if you run the experiments here in Google Colab.

### Helper code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install blobfile mpi4py

In [ ]:
# Clone and install guided-diffusion (run once; skip if already on Drive)
import os
GUIDED_DIFF_DIR = '/content/drive/MyDrive/guided-diffusion'
if not os.path.exists(GUIDED_DIFF_DIR):
    !git clone https://github.com/openai/guided-diffusion.git {GUIDED_DIFF_DIR}
%cd {GUIDED_DIFF_DIR}
!pip install -e . -q
!pip install blobfile mpi4py -q
os.makedirs(f'{GUIDED_DIFF_DIR}/models', exist_ok=True)

#### Change the directory (TODO)

In [ ]:
directory = '/content/drive/MyDrive/path/to/guided-diffusion/repo'

In [ ]:
!wget https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_classifier.pt -P {directory}/models
!wget https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt -P {directory}/models
%cd {directory}

#### Example code for running a script

In [ ]:
# define flags
SAMPLE_FLAGS = """
    --batch_size 1
    --num_samples 2
    --timestep_respacing 250
"""
MODEL_FLAGS = """
    --attention_resolutions 32,16,8
    --class_cond False
    --diffusion_steps 1000
    --image_size 256
    --learn_sigma True
    --noise_schedule linear
    --num_channels 256
    --num_head_channels 64
    --num_res_blocks 2
    --resblock_updown True
    --use_fp16 True
    --use_scale_shift_norm True
"""

# run a script
!python scripts/classifier_sample.py {MODEL_FLAGS.replace('\n', '')} \
    --classifier_scale 10.0 \
    --classifier_path models/256x256_classifier.pt \
    --model_path models/256x256_diffusion_uncond.pt \
    --save_dir /content/test_run \
    {SAMPLE_FLAGS.replace('\n', '')}

### A: unconditional generation

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt

# --- A: Unconditional generation ---
os.environ['OPENAI_LOGDIR'] = '/content/output_uncond'
os.makedirs('/content/output_uncond', exist_ok=True)

MODEL_FLAGS_UNCOND = (
    "--attention_resolutions 32,16,8 --class_cond False --diffusion_steps 1000 "
    "--image_size 256 --learn_sigma True --noise_schedule linear "
    "--num_channels 256 --num_head_channels 64 --num_res_blocks 2 "
    "--resblock_updown True --use_fp16 True --use_scale_shift_norm True"
)
SAMPLE_FLAGS_UNCOND = "--batch_size 4 --num_samples 4 --timestep_respacing 250"

!python {directory}/scripts/image_sample.py {MODEL_FLAGS_UNCOND} \
    --model_path {directory}/models/256x256_diffusion_uncond.pt \
    {SAMPLE_FLAGS_UNCOND}

npz_files = sorted(glob.glob('/content/output_uncond/*.npz'))
samples = np.load(npz_files[0])['arr_0']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, ax in enumerate(axes):
    ax.imshow(samples[i])
    ax.axis('off')
plt.suptitle('Part 5A: Unconditional generation (256×256)', fontsize=14)
plt.tight_layout()
plt.savefig('part5A_unconditional.png', dpi=100, bbox_inches='tight')
plt.show()

### B: progression over time visualization

In [ ]:
import sys
import torch as th
sys.path.insert(0, directory)
from guided_diffusion import dist_util
from guided_diffusion.script_util import create_model_and_diffusion

# Load model once for B and C experiments
model_args = dict(
    image_size=256, num_channels=256, num_res_blocks=2,
    num_heads=4, num_heads_upsample=-1, num_head_channels=64,
    attention_resolutions="32,16,8", channel_mult="", dropout=0.0,
    class_cond=False, use_checkpoint=False, use_scale_shift_norm=True,
    resblock_updown=True, use_fp16=True, use_new_attention_order=False,
    learn_sigma=True, diffusion_steps=1000, noise_schedule="linear",
    timestep_respacing="250", use_kl=False, predict_xstart=False,
    rescale_timesteps=False, rescale_learned_sigmas=False,
)
model_prog, diffusion_prog = create_model_and_diffusion(**model_args)
model_prog.load_state_dict(
    dist_util.load_state_dict(f'{directory}/models/256x256_diffusion_uncond.pt', map_location='cpu')
)
model_prog = model_prog.cuda().eval()
if model_args['use_fp16']:
    model_prog.convert_to_fp16()

# --- B: Denoising progression over time ---
# Reverse denoising: iterate from t=T-1 down to t=0 (high noise → clean)
noise = th.randn(1, 3, 256, 256, device='cuda')
intermediates = {}
img = noise.clone()
indices = list(range(diffusion_prog.num_timesteps))[::-1]  # [249, 248, ..., 0]

for step_num, i in enumerate(indices):
    t_val = th.tensor([i], device='cuda')
    with th.no_grad():
        out = diffusion_prog.p_sample(model_prog, img, t_val, clip_denoised=True, model_kwargs={})
    img = out['sample']
    if step_num % 25 == 0 or step_num == len(indices) - 1:
        intermediates[step_num] = out['pred_xstart'][0].cpu().float()

steps_to_show = sorted(intermediates.keys())
fig, axes = plt.subplots(1, len(steps_to_show), figsize=(22, 3))
for ax, step in zip(axes, steps_to_show):
    img_np = intermediates[step].permute(1, 2, 0).numpy()
    img_np = (img_np.clip(-1, 1) + 1) / 2
    ax.imshow(img_np)
    ax.set_title(f'step {step}', fontsize=8)
    ax.axis('off')
plt.suptitle('Part 5B: Denoising progression over time', fontsize=13)
plt.tight_layout()
plt.savefig('part5B_progression.png', dpi=100, bbox_inches='tight')
plt.show()

### C: interpolation visualization

In [ ]:
# --- C: Spherical interpolation in noise space ---
def slerp(z0, z1, alpha):
    z0f, z1f = z0.view(-1), z1.view(-1)
    dot = (z0f * z1f).sum() / (z0f.norm() * z1f.norm() + 1e-8)
    dot = dot.clamp(-1, 1)
    theta = th.acos(dot)
    if theta.abs() < 1e-6:
        return (1 - alpha) * z0 + alpha * z1
    return (th.sin((1 - alpha) * theta) / th.sin(theta)) * z0 + \
           (th.sin(alpha * theta) / th.sin(theta)) * z1

n_interp = 8
z0 = th.randn(1, 3, 256, 256, device='cuda')
z1 = th.randn(1, 3, 256, 256, device='cuda')
alphas = th.linspace(0, 1, n_interp)

interp_images = []
for alpha in alphas:
    z_interp = slerp(z0, z1, alpha.item())
    with th.no_grad():
        sample = diffusion_prog.p_sample_loop(
            model_prog, (1, 3, 256, 256),
            noise=z_interp,
            clip_denoised=True,
            progress=False,
            model_kwargs={},
        )
    interp_images.append(sample[0].cpu().float())

fig, axes = plt.subplots(1, n_interp, figsize=(22, 3))
for ax, img_t, alpha in zip(axes, interp_images, alphas):
    img_np = img_t.permute(1, 2, 0).numpy()
    img_np = (img_np.clip(-1, 1) + 1) / 2
    ax.imshow(img_np)
    ax.set_title(f'α={alpha:.2f}', fontsize=8)
    ax.axis('off')
plt.suptitle('Part 5C: Spherical interpolation between two noise vectors', fontsize=13)
plt.tight_layout()
plt.savefig('part5C_interpolation.png', dpi=100, bbox_inches='tight')
plt.show()

### D: conditional generation

In [ ]:
# --- D: Conditional generation with classifier guidance ---
os.environ['OPENAI_LOGDIR'] = '/content/output_cond'
os.makedirs('/content/output_cond', exist_ok=True)

# Download class-conditional model weights
!wget -q https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion.pt \
    -P {directory}/models

MODEL_FLAGS_COND = (
    "--attention_resolutions 32,16,8 --class_cond True --diffusion_steps 1000 "
    "--image_size 256 --learn_sigma True --noise_schedule linear "
    "--num_channels 256 --num_head_channels 64 --num_res_blocks 2 "
    "--resblock_updown True --use_fp16 True --use_scale_shift_norm True"
)

!python {directory}/scripts/classifier_sample.py \
    {MODEL_FLAGS_COND} \
    --classifier_scale 10.0 \
    --classifier_path {directory}/models/256x256_classifier.pt \
    --model_path {directory}/models/256x256_diffusion.pt \
    --batch_size 8 --num_samples 8 --timestep_respacing 250

npz_cond = sorted(glob.glob('/content/output_cond/*.npz'))[0]
cond_samples = np.load(npz_cond)['arr_0']

fig, axes = plt.subplots(1, min(8, len(cond_samples)), figsize=(20, 3))
for i, ax in enumerate(axes):
    ax.imshow(cond_samples[i])
    ax.axis('off')
plt.suptitle('Part 5D: Conditional generation (classifier scale=10)', fontsize=13)
plt.tight_layout()
plt.savefig('part5D_conditional.png', dpi=100, bbox_inches='tight')
plt.show()

### E: conditional generation with different classifier scales

In [ ]:
# --- E: Conditional generation with different classifier scales ---
scales = [0.0, 1.0, 5.0, 10.0, 25.0]
scale_images = {}

for scale in scales:
    out_dir = f'/content/output_scale_{scale}'
    os.makedirs(out_dir, exist_ok=True)
    os.environ['OPENAI_LOGDIR'] = out_dir

    !python {directory}/scripts/classifier_sample.py \
        {MODEL_FLAGS_COND} \
        --classifier_scale {scale} \
        --classifier_path {directory}/models/256x256_classifier.pt \
        --model_path {directory}/models/256x256_diffusion.pt \
        --batch_size 4 --num_samples 4 --timestep_respacing 250

    npz = sorted(glob.glob(f'{out_dir}/*.npz'))[0]
    scale_images[scale] = np.load(npz)['arr_0']

n_show = 4
fig, axes = plt.subplots(len(scales), n_show, figsize=(16, len(scales) * 3))
for row, scale in enumerate(scales):
    for col in range(n_show):
        axes[row, col].imshow(scale_images[scale][col])
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'scale={scale}', fontsize=11, rotation=0, labelpad=65)
plt.suptitle('Part 5E: Effect of classifier guidance scale', fontsize=14)
plt.tight_layout()
plt.savefig('part5E_scale_sweep.png', dpi=100, bbox_inches='tight')
plt.show()

### F: extra credit (explore freely)